# Uber ride data cleaning
This notebook cleans and prepares Uber ride data for further analysis and dataset merging.

Initial exploratory cleaning and inspection were previously performed manually in Excel to better understand the dataset structure and identify potential issues.
The cleaning steps are reproduced programmatically in Python to ensure reproducibility and documentation of the workflow.

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('../../data/raw_data/driver_lifetime_trips-0.csv')

## Dataset overview
The initial inspection is performed to understand the dataset structure , data types, and potential missing values.

In [5]:
df.head()

,global_product_name,status,request_timestamp_local,begintrip_timestamp_local,dropoff_timestamp_local,trip_distance_miles,trip_duration_seconds,base_fare_local,original_fare_local,cancellation_fee_local,currency_code,vehicle_uuid,license_plate
0,UberX,completed,2024-04-19T13:04:18.000Z,2024-04-19T13:06:29.000Z,2024-04-19T13:21:28.000Z,2.726422,899.0,1.43,5.34,0.0,EUR,2c25a94f-8e7e-484b-a17e-954850e64b54,NaN
1,UberX,completed,2024-04-19T14:40:02.000Z,2024-04-19T14:47:35.000Z,2024-04-19T14:55:15.000Z,1.818851,460.0,1.43,4.11,0.0,EUR,2c25a94f-8e7e-484b-a17e-954850e64b54,NaN
2,UberX,completed,2024-05-01T12:40:49.000Z,2024-05-01T12:47:15.000Z,2024-05-01T13:05:28.000Z,1.584360,1086.0,1.43,4.00,0.0,EUR,2c25a94f-8e7e-484b-a17e-954850e64b54,NaN
3,UberX,rider_canceled,2024-05-01T12:52:54.000Z,NaN,NaN,0.000000,NaN,NaN,0.00,0.0,EUR,2c25a94f-8e7e-484b-a17e-954850e64b54,NaN
4,UberX,completed,2024-05-01T13:02:55.000Z,2024-05-01T13:12:16.000Z,2024-05-01T13:25:53.000Z,5.166890,817.0,1.43,8.12,0.0,EUR,2c25a94f-8e7e-484b-a17e-954850e64b54,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1564 entries, 0 to 1563
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   global_product_name        1564 non-null   object 
 1   status                     1564 non-null   object 
 2   request_timestamp_local    1564 non-null   object 
 3   begintrip_timestamp_local  1345 non-null   object 
 4   dropoff_timestamp_local    1342 non-null   object 
 5   trip_distance_miles        1564 non-null   float64
 6   trip_duration_seconds      1358 non-null   float64
 7   base_fare_local            1352 non-null   float64
 8   original_fare_local        1558 non-null   float64
 9   cancellation_fee_local     1564 non-null   float64
 10  currency_code              1558 non-null   object 
 11  vehicle_uuid               1564 non-null   object 
 12  license_plate              1515 non-null   object 
dtypes: float64(5), object(8)
memory usage: 159.0+ KB

In [7]:
df.shape

(1564, 13)

In [8]:
df.isnull().sum()

global_product_name            0
status                         0
request_timestamp_local        0
begintrip_timestamp_local    219
dropoff_timestamp_local      222
trip_distance_miles            0
trip_duration_seconds        206
base_fare_local              212
original_fare_local            6
cancellation_fee_local         0
currency_code                  6
vehicle_uuid                   0
license_plate                 49
dtype: int64

### Observations
Rows containing missing trip and fare values will be removed, as they are likely cancelled or incomplete rides amd are not useful for revenue analysis.

Datetime columns are currently stored as object type and will require conversion to datetime format for time-based analysis.


## Removing irrelevant columns
Columns containing identifiers, redundant information, or variables not relevant for ride demand analysis will be removed.

In [9]:
df = df.drop(columns=['cancellation_fee_local','currency_code','vehicle_uuid','license_plate'])

In [10]:
print(df.columns)


Index(['global_product_name', 'status', 'request_timestamp_local',
       'begintrip_timestamp_local', 'dropoff_timestamp_local',
       'trip_distance_miles', 'trip_duration_seconds', 'base_fare_local',
       'original_fare_local'],
      dtype='object')


In [11]:
df['status'].unique()

array(['completed', 'rider_canceled', 'fare_split', 'driver_canceled',
       'unfulfilled'], dtype=object)

In [12]:
df['status'].value_counts()

status
completed          1334
rider_canceled      213
fare_split            8
unfulfilled           6
driver_canceled       3
Name: count, dtype: int64

## Filtering completed rides
Only rides with a 'completed' status were retained for analysis.

Cancelled and similar rides were removed because they would just distort revenue and demand analysis.

In [13]:
df = df[df['status']=='completed']

In [14]:
df = df.drop(columns=['status'])

In [15]:
df.columns

Index(['global_product_name', 'request_timestamp_local',
       'begintrip_timestamp_local', 'dropoff_timestamp_local',
       'trip_distance_miles', 'trip_duration_seconds', 'base_fare_local',
       'original_fare_local'],
      dtype='object')

## Renaming columns
Column names were simplified and standardized to improve readability.

In [16]:
df = df.rename(columns={
    'global_product_name':'ride_category',
    'request_timestamp_local':'request_time',
    'begintrip_timestamp_local':'start_time',
    'dropoff_timestamp_local':'end_time',
    'trip_distance_miles':'trip_distance_km',
    'trip_duration_seconds':'trip_duration_minutes',
    'base_fare_local':'base_fare',
    'original_fare_local':'original_fare'
})

In [17]:
df.columns

Index(['ride_category', 'request_time', 'start_time', 'end_time',
       'trip_distance_km', 'trip_duration_minutes', 'base_fare',
       'original_fare'],
      dtype='object')

## Converting datetime columns
Datetime columns were converted from object type to datetime format.

In [18]:
df['request_time'] = pd.to_datetime(df['request_time'])
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1334 entries, 0 to 1561
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   ride_category          1334 non-null   object             
 1   request_time           1334 non-null   datetime64[ns, UTC]
 2   start_time             1334 non-null   datetime64[ns, UTC]
 3   end_time               1334 non-null   datetime64[ns, UTC]
 4   trip_distance_km       1334 non-null   float64            
 5   trip_duration_minutes  1334 non-null   float64            
 6   base_fare              1334 non-null   float64            
 7   original_fare          1334 non-null   float64            
dtypes: datetime64[ns, UTC](3), float64(4), object(1)
memory usage: 93.8+ KB


## Missing values check
After filtering only completed rides, the dataset is checked for remaining missing values.

In [20]:
df.isnull().sum()

ride_category            0
request_time             0
start_time               0
end_time                 0
trip_distance_km         0
trip_duration_minutes    0
base_fare                0
original_fare            0
dtype: int64

No remaining missing values were found in the dataset.

## Duplicate values check
The dataset is checked for duplicate rows to ensure data consistency and avoid duplicated trip records.

In [21]:
df.duplicated().sum()

np.int64(0)

No duplicate rows were found in the dataset.

## Unit conversion
Trip distance and duration values were converted into more interpretable units for analysis.
- Distance was converted from miles to kilometers
- Duration was converted from seconds to minutes

In [22]:
df['trip_distance_km'] = df['trip_distance_km'] * 1.60934
df['trip_duration_minutes'] = df['trip_duration_minutes'] / 60

df['trip_distance_km'] = df['trip_distance_km'].round(2)
df['trip_duration_minutes'] = df['trip_duration_minutes'].round(2)

## Feature engineering 
Additional time-based features were created from datetime columns to support demand and seasonality analysis.

In [23]:
df['hour_of_day'] = df['start_time'].dt.hour
df['day_of_week'] = df['start_time'].dt.day_name()
df['month'] = df['start_time'].dt.month_name()

In [24]:
df['is_weekend'] = df['day_of_week'].isin(['Saturday','Sunday'])

In [25]:
df.head()

,ride_category,request_time,start_time,end_time,trip_distance_km,trip_duration_minutes,base_fare,original_fare,hour_of_day,day_of_week,month,is_weekend
0,UberX,2024-04-19 13:04:18+00:00,2024-04-19 13:06:29+00:00,2024-04-19 13:21:28+00:00,4.39,14.98,1.43,5.34,13,Friday,April,False
1,UberX,2024-04-19 14:40:02+00:00,2024-04-19 14:47:35+00:00,2024-04-19 14:55:15+00:00,2.93,7.67,1.43,4.11,14,Friday,April,False
2,UberX,2024-05-01 12:40:49+00:00,2024-05-01 12:47:15+00:00,2024-05-01 13:05:28+00:00,2.55,18.10,1.43,4.00,12,Wednesday,May,False
4,UberX,2024-05-01 13:02:55+00:00,2024-05-01 13:12:16+00:00,2024-05-01 13:25:53+00:00,8.32,13.62,1.43,8.12,13,Wednesday,May,False
5,UberX,2024-05-01 13:21:49+00:00,2024-05-01 13:40:32+00:00,2024-05-01 13:55:00+00:00,5.34,14.47,1.43,5.67,13,Wednesday,May,False


## Exporting cleaned dataset

In [28]:
df.to_csv('../../data/processed_data/uber_rides_cleaned.csv', index = False)